# OOD 결과 테이블 생성

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 📋 실행 순서

### Option A: 이미 CSV 파일이 있는 경우 (권장)
1. **셀 1-6**을 순서대로 실행

### Option B: WandB에서 최신 데이터를 가져오는 경우
1. **셀 0-1, 0-2, 0-3**을 실행하여 WandB에서 데이터 수집
   - ⚠️ **주의**: 시간이 오래 걸릴 수 있습니다 (1-5분)
   - `results/finished_runs_data.csv` 파일이 생성/업데이트됩니다
2. **셀 1-6**을 순서대로 실행

---

## 📑 셀별 역할 요약

| 셀 | 역할 |
|----|------|
| **0-1** | WandB API 초기화, project_name, sweep_id 설정 |
| **0-2** | WandB에서 run 데이터 수집 (config, summary, history), Task/forgetting 계산, runs_df 생성 |
| **0-2 결과** | runs_df.head()로 수집 결과 미리보기 |
| **분포 확인** | fusion_type, increment 고유값 출력 |
| **0-3** | runs_df를 CSV로 저장 (all_runs, finished_runs) |
| **1. 설정** | preset 선택 → cfg (datasets, increments, fusion_type, cl_methods 등) 설정 |
| **1. 로드** | CSV에서 df 로드 |
| **2. 파싱** | sweep_name에서 model, modality, dataset 추출 |
| **3. 함수** | create_ood_result_table, save_ood_table_to_excel 정의 |
| **4. 생성** | cfg 기반으로 테이블 생성 및 엑셀 저장 |
| **진단** | cfg 조건에 맞는 데이터 존재 여부 확인 |

---

## 0. WandB에서 데이터 수집 (선택사항)

**이미 `results/finished_runs_data.csv` 파일이 있다면 이 섹션을 건너뛰고 셀 1부터 시작하세요!**

### 💡 특정 Sweep만 불러오기
셀 0-1에서 `sweep_id`를 지정하면 해당 sweep의 runs만 불러옵니다 (전체 프로젝트보다 훨씬 빠름).
- 예: `sweep_id = "abc123xy"`
- Sweep ID는 WandB 웹 UI의 sweep URL에서 확인 가능합니다.


In [27]:
# =============================================================================
# [셀 0-1] WandB API 초기화
# =============================================================================
# • wandb.Api(): WandB 클라우드에 연결하여 프로젝트/run 데이터를 가져올 수 있는 API 객체 생성
# • project_name: "entity/project" 형식. WandB 웹에서 프로젝트 URL 확인 가능
# • sweep_id: 특정 sweep만 불러올 때 사용. None이면 프로젝트 전체 runs 조회 (느림)
#    - sweep_id 지정 시 해당 sweep의 runs만 가져와 훨씬 빠름
#    - Sweep ID는 WandB sweep 페이지 URL의 마지막 부분 (예: .../sweeps/abc123xy)
# =============================================================================

import wandb
from pathlib import Path

api = wandb.Api()
project_name = "mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)"

# 특정 sweep만 불러오려면 sweep_id를 지정하세요. None이면 프로젝트 전체 runs를 불러옵니다.
sweep_id = "l49po2dj"  # 예: "abc123xy"  → 해당 sweep의 runs만 불러옴

print(f"📊 연결된 프로젝트: {project_name}")
print(f"📌 Sweep ID: {sweep_id if sweep_id else '(전체 프로젝트)'}")
print("✅ WandB API 초기화 완료")


📊 연결된 프로젝트: mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)
📌 Sweep ID: l49po2dj
✅ WandB API 초기화 완료


In [28]:
# =============================================================================
# [셀 0-2] WandB에서 Run 데이터 수집
# =============================================================================
# 이 셀은 다음을 수행합니다:
#
# 1. _get_task_acc_cols(): history 컬럼 중 Task/[XX-YY]_acc 패턴 추출
#    - WandB에 Task/[00-01]_acc, Task/[02-03]_acc 등으로 기록됨
#    - task 번호 순으로 정렬하여 반환
#
# 2. _get_history_data(): run.history() 또는 scan_history()로 step별 기록 가져오기
#    - summary에는 Task/[xx-xx]_acc가 NaN인 경우가 있어, history의 마지막 행 사용
#
# 3. _compute_forgetting(): CL forgetting 지표 계산 (mammoth 방식)
#    - 각 task의 max(과거 acc) - 최종 acc를 평균
#
# 4. get_all_runs_from_project(): 각 run의 config + summary + history에서
#    Task/[xx-xx]_acc 채우기 + forgetting 계산 → runs_data 리스트로 반환
#
# 5. runs_df: 전체 데이터를 DataFrame으로 생성 (다음 셀에서 저장/분석에 사용)
# =============================================================================

import pandas as pd
import numpy as np
import re

def _get_task_acc_cols(hist_or_row):
    """Task/[XX-YY]_acc 패턴 컬럼명 추출 (history 컬럼 또는 dict keys)"""
    keys = hist_or_row.columns if hasattr(hist_or_row, 'columns') else hist_or_row.keys()
    acc = [k for k in keys if isinstance(k, str) and k.startswith('Task/[') and k.endswith(']_acc') and 'NME' not in k]
    return sorted(acc, key=lambda x: int(re.search(r'[(\[]?(\d+)-', str(x)).group(1)) if re.search(r'[(\[]?(\d+)-', str(x)) else 0)

def _get_history_data(run):
    """run.history() 또는 scan_history()로 데이터 수집. (hist, acc_cols) 반환"""
    try:
        hist = run.history(samples=5000)
        if hist is not None and len(hist) >= 1:
            acc_cols = _get_task_acc_cols(hist)
            if acc_cols:
                return hist, acc_cols
    except Exception:
        pass
    try:
        rows = list(run.scan_history())
        if rows:
            acc_cols = _get_task_acc_cols(rows[0])
            if acc_cols:
                return pd.DataFrame(rows), acc_cols
    except Exception:
        pass
    return None, []

def _compute_forgetting(hist, acc_cols):
    """(hist, acc_cols)에서 Forgetting 계산"""
    if hist is None or len(acc_cols) < 2:
        return np.nan
    n_tasks = len(acc_cols)
    if 'Task/Task_ID' in hist.columns:
        h = hist.dropna(subset=['Task/Task_ID']).sort_values('Task/Task_ID').tail(n_tasks)
    else:
        h = hist.dropna(subset=[acc_cols[0]], how='all').tail(n_tasks)
    if len(h) < 2:
        return np.nan
    results = []
    for t, (_, row) in enumerate(h.iterrows()):
        vals = [float(row[c]) if pd.notna(row.get(c)) else 0.0 for c in acc_cols[:t+1]]
        results.append(vals + [0.0] * (n_tasks - len(vals)))
    np_res = np.array(results)
    max_acc = np.max(np_res, axis=0)
    return float(np.mean([max_acc[i] - results[-1][i] for i in range(n_tasks - 1)]))

def get_all_runs_from_project(project_name, sweep_id=None, filters=None, compute_forgetting=True, use_history_for_task_acc=True):
    """프로젝트의 Run 데이터를 가져옵니다.
    
    Args:
        project_name: "entity/project" 형식의 프로젝트 경로
        sweep_id: 특정 sweep의 ID. 지정하면 해당 sweep의 runs만 불러옴. None이면 프로젝트 전체.
        filters: api.runs()에 전달할 추가 필터 (sweep_id 없을 때만 사용)
        compute_forgetting: True이면 각 run의 history에서 Forgetting 계산
    """
    api = wandb.Api()
    
    if sweep_id:
        sweep_path = f"{project_name}/{sweep_id}"
        print(f"🔍 Sweep '{sweep_path}'에서 Run 데이터를 가져오는 중...")
        sweep = api.sweep(sweep_path)
        runs = sweep.runs
    else:
        runs = api.runs(project_name, filters=filters) if filters else api.runs(project_name)
        print(f"🔍 프로젝트 전체 Run 데이터를 가져오는 중...")
    
    runs_data = []
    for i, run in enumerate(runs, 1):
        if i % 10 == 0:
            print(f"   진행중: {i} runs 처리...")
        
        run_info = {
            'run_id': run.id,
            'run_name': run.name,
            'state': run.state,
            'sweep_id': run.sweep.id if run.sweep else None,
            'sweep_name': run.sweep.name if run.sweep else None,
            'created_at': run.created_at,
        }
        run_info.update(run.config)
        run_info.update(run.summary._json_dict)
        hist, acc_cols = _get_history_data(run)
        if use_history_for_task_acc and hist is not None and acc_cols:
            last_row = hist.dropna(subset=acc_cols, how='all').tail(1)
            if len(last_row) > 0:
                for c in acc_cols:
                    v = last_row[c].iloc[0]
                    if pd.notna(v):
                        run_info[c] = float(v)
        if compute_forgetting:
            run_info['Task/forgetting'] = _compute_forgetting(hist, acc_cols)
        runs_data.append(run_info)
    
    print(f"✅ 총 {len(runs_data)}개의 Run 데이터 수집 완료")
    return runs_data

# 데이터 수집 실행
print("=" * 100)
print("🔄 WandB에서 데이터 수집 시작")
print("=" * 100)
if sweep_id:
    print(f"📌 특정 Sweep만 수집: {sweep_id}")
else:
    print("⚠️ 시간이 다소 걸릴 수 있습니다 (run 개수에 따라 1-5분)")
print()

all_runs_data = get_all_runs_from_project(project_name, sweep_id=sweep_id)
runs_df = pd.DataFrame(all_runs_data)

print(f"\n" + "=" * 100)
print(f"📊 데이터 수집 완료")
print("=" * 100)
print(f"총 Run 개수: {len(runs_df)}")
print(f"총 컬럼 개수: {len(runs_df.columns)}")


🔄 WandB에서 데이터 수집 시작
📌 특정 Sweep만 수집: l49po2dj

🔍 Sweep 'mmea-owcl/Experimental Results on the MMEA-OWCL (Evaluation CL & OOD)/l49po2dj'에서 Run 데이터를 가져오는 중...
   진행중: 10 runs 처리...
✅ 총 15개의 Run 데이터 수집 완료

📊 데이터 수집 완료
총 Run 개수: 15
총 컬럼 개수: 251


In [29]:
# =============================================================================
# [셀 0-2 결과 확인] 수집된 runs_df 미리보기
# =============================================================================
# • runs_df: 0-2 셀에서 수집한 모든 run의 config, summary, Task/[xx-xx]_acc, forgetting 등
# • head(): 처음 5행만 출력하여 컬럼 구성과 값 분포 확인
# =============================================================================
runs_df.head()

,run_id,run_name,state,sweep_id,sweep_name,created_at,lr,arch,host,mode,...,Task/[00-01]_acc,Task/[02-03]_acc,Task/[04-05]_acc,Task/[06-07]_acc,Task/[08-09]_acc,Task/[10-11]_acc,Task/[12-13]_acc,Task/[14-15]_acc,Task/[16-17]_acc,Task/[18-19]_acc
0,4eeb5d29,tbn_mand_4eeb5d29,finished,l49po2dj,Evaluate TBN-MAND (All) Method on DataEgo (See...,2026-03-23T04:51:20Z,0.001,BNInception,1e65b8e7e400,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,55d88c34,tbn_mand_55d88c34,finished,l49po2dj,Evaluate TBN-MAND (All) Method on DataEgo (See...,2026-03-23T04:51:30Z,0.001,BNInception,1e65b8e7e400,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7f59cf60,tbn_mand_7f59cf60,finished,l49po2dj,Evaluate TBN-MAND (All) Method on DataEgo (See...,2026-03-23T04:51:40Z,0.001,BNInception,1e65b8e7e400,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,15349cbf,tbn_mand_15349cbf,finished,l49po2dj,Evaluate TBN-MAND (All) Method on DataEgo (See...,2026-03-23T04:51:50Z,0.001,BNInception,1e65b8e7e400,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,63c74c0e,tbn_mand_63c74c0e,finished,l49po2dj,Evaluate TBN-MAND (All) Method on DataEgo (See...,2026-03-23T04:51:57Z,0.001,BNInception,1e65b8e7e400,eval,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
# =============================================================================
# [데이터 분포 확인] fusion_type, increment 고유값 출력
# =============================================================================
# • fusion_type: 모달리티 결합 방식 (예: mand_fusion, concat, auxiliary_head 등)
# • increment: CL task 수 (예: 10, 5, 2 → 데이터셋을 10/5/2개 구간으로 나눔)
# → 이후 테이블 생성 시 어떤 fusion_type, increment 조합을 사용할지 참고
# =============================================================================
print("1. fusion_type: ", runs_df["fusion_type"].unique())
print("2. increment: ", runs_df["increment"].unique())

1. fusion_type:  ['mand_fusion']
2. increment:  [10  5  2]


In [31]:
# =============================================================================
# [셀 0-3] 수집한 데이터를 CSV 파일로 저장
# =============================================================================
# • output_dir: notebook/results/ (노트북 위치 기준)
# • all_runs_file: 전체 run (running 포함) → all_runs_data{sweep_id}.csv
# • finished_runs_file: state=='finished'인 run만 → finished_runs_data{sweep_id}.csv
# • sweep_id가 있으면 파일명에 sweep_id 붙여 여러 sweep 결과를 분리 저장
# • finished_runs_filename: 다음 섹션(데이터 로드)에서 사용할 파일명 변수
# =============================================================================

import os
# 노트북 파일이 있는 디렉토리를 기준으로 경로 설정
notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'
output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

# sweep_id가 있으면 sweep별로 파일명 구분 (여러 sweep 결과를 보관 가능)
suffix = f"_{sweep_id}" if sweep_id else ""
all_runs_file = f"all_runs_data{suffix}.csv"
finished_runs_file = f"finished_runs_data{suffix}.csv"

# 전체 데이터 저장
output_file = output_dir / all_runs_file
runs_df.to_csv(output_file, index=False)

# 완료된 run만 필터링하여 저장
finished_runs = runs_df[runs_df['state'] == 'finished']

print("=" * 100)
print("💾 CSV 파일 저장 완료")
print("=" * 100)
print(f"✅ 전체 데이터: {output_file}")
print(f"   - 총 Run 개수: {len(runs_df)}")
print(f"   - 파일 크기: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
if sweep_id:
    print(f"   - Sweep ID: {sweep_id}")

if len(finished_runs) > 0:
    finished_output = output_dir / finished_runs_file
    finished_runs.to_csv(finished_output, index=False)
    print(f"\n✅ 완료된 Run 파일: {finished_output}")
    print(f"   - 완료된 Run 개수: {len(finished_runs)}")
else:
    print(f"\n⚠️ 완료된(finished) Run이 없습니다. 아래 테이블 생성 시 먼저 완료된 Run을 수집하세요.")

# 다음 셀(데이터 로드)에서 사용할 파일명
finished_runs_filename = finished_runs_file

print("\n✨ 데이터 수집 및 저장 완료!")
print("👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.")


💾 CSV 파일 저장 완료
✅ 전체 데이터: /workspace/MMEA-MAND/notebook/results/all_runs_data_l49po2dj.csv
   - 총 Run 개수: 15
   - 파일 크기: 0.05 MB
   - Sweep ID: l49po2dj

✅ 완료된 Run 파일: /workspace/MMEA-MAND/notebook/results/finished_runs_data_l49po2dj.csv
   - 완료된 Run 개수: 15

✨ 데이터 수집 및 저장 완료!
👉 이제 아래 셀 1부터 실행하여 OOD 결과 테이블을 생성하세요.


## 1. 분석 설정 (동적 변경 가능)

아래 설정 셀에서 **preset** 또는 **수동 설정**을 변경한 뒤 데이터 로드 → 파싱 → 테이블 생성을 실행하세요.

**preset 예시:**
- `preset = "dataego_mand"` → `1.dataego-mand-herding.yaml` 실행 결과 (MAND, mand_fusion, DataEgo)
- `preset = "dataego_replay"` → Replay 방식 결과 (PRESETS에서 주석 해제 후 사용)

In [32]:
# =============================================================================
# [셀 1. 분석 설정] preset 또는 수동 설정
# =============================================================================
# • preset: "dataego_mand" → 1.dataego-mand-herding.yaml 결과 분석
#           "dataego_replay" → Replay 방식 결과 분석 (주석 해제 후 사용)
#           None → PRESETS["dataego_mand"] 기본값 사용
#
# • PRESETS: dataset, increment, modality, seed, fusion_type, cl_methods, ood_methods
#   - target_datasets: 분석할 데이터셋 (DataEgo, UESTC-MMEA-CL 등)
#   - target_increments: [10, 5, 2] 등
#   - target_fusion_types: mand_fusion, concat 등
#   - ood_methods: None이면 create_ood_result_table에서 fusion_type에 따라 자동 선택
#
# • cfg: 위 preset에서 선택된 설정 딕셔너리 (이후 테이블 생성에 사용)
# • load_sweep_id: None이면 0-3에서 저장한 finished_runs_filename 사용
# =============================================================================

preset = "dataego_mand"  # dataego_replay | None(수동)

PRESETS = {
    "dataego_mand": {
        "target_datasets": ["DataEgo"], "target_increments": [10, 5, 2], "target_modality": "All",
        "target_seeds": [1993, 1994, 1995, 1996, 1997], "target_fusion_types": ["mand_fusion"],
        "cl_methods": ["MAND"],
        "ood_methods": ["MaxLogit_Baseline", "MaxLogit_Hybrid_UniformSum", "MaxLogit_Hybrid_UniformAverage", "MaxLogit_Hybrid_ConfNormalized"],
    },
    # "dataego_replay": {
    #     "target_datasets": ["DataEgo"], "target_increments": [10, 5, 2], "target_modality": "All",
    #     "target_seeds": [1993, 1994, 1995, 1996, 1997], "target_fusion_types": ["concat"],
    #     "cl_methods": ["Replay"], "ood_methods": None,
    # },
}

cfg = (PRESETS.get(preset, PRESETS["dataego_mand"]) if preset else PRESETS["dataego_mand"]).copy()
load_sweep_id = None
print(f"✅ Preset: {preset or 'manual'}")

✅ Preset: dataego_mand


---

## 1. OOD 결과 분석 시작

**분석 설정 셀을 먼저 실행**한 뒤, 데이터 로드 → 파싱 → 테이블 생성을 순서대로 실행.


# OOD 결과 테이블 생성 (분석 파이프라인)

WandB에서 수집한 OOD 평가 결과를 엑셀 테이블로 생성합니다.

## 실행 순서 (반드시 순서대로)
1. **분석 설정** — preset 변경 후 실행하여 cfg 설정
2. **데이터 로드** — CSV에서 df 읽기
3. **Sweep Name 파싱** — model, modality, dataset 컬럼 추가
4. **함수 로드** — create_ood_result_table, save_ood_table_to_excel
5. **테이블 생성 및 저장** — cfg 기반 필터링 → 엑셀 파일 생성

In [33]:
# =============================================================================
# [셀 1. 데이터 로드] CSV에서 Run 데이터 읽기
# =============================================================================
# 파일 경로 우선순위:
#   1) finished_runs_filename (0-3 셀에서 저장한 파일명)
#   2) load_sweep_id가 있으면: finished_runs_data_{load_sweep_id}.csv
#   3) 기본: finished_runs_data.csv
#
# • notebook_dir: notebook/ 폴더 경로 (cwd가 notebook이 아니면 /notebook 추가)
# • df: 로드된 DataFrame. 이후 parse, create_ood_result_table 등에서 사용
# =============================================================================

import pandas as pd
import numpy as np
import re
import os
from pathlib import Path

notebook_dir = Path(os.getcwd()) if 'notebook' in os.getcwd() else Path(os.getcwd()) / 'notebook'

# 0-3에서 저장한 파일 사용 (sweep_id 있으면 sweep별 파일, 없으면 기본 파일)
try:
    csv_filename = finished_runs_filename
except NameError:
    try:
        csv_filename = f"finished_runs_data_{load_sweep_id}.csv" if load_sweep_id else "finished_runs_data.csv"
    except NameError:
        csv_filename = "finished_runs_data.csv"
csv_path = notebook_dir / 'results' / csv_filename

print(f"📂 데이터 파일 경로: {csv_path}")
print(f"   파일 존재 여부: {csv_path.exists()}\n")

if not csv_path.exists():
    print("⚠️  파일을 찾을 수 없습니다!")
    print("👉 먼저 셀 0-1, 0-2, 0-3을 실행하여 데이터를 수집하세요.")
else:
    df = pd.read_csv(csv_path)
    print(f"✅ 데이터 로드 완료: {len(df)} runs")
    print(f"   • 컬럼 수: {len(df.columns)}")

📂 데이터 파일 경로: /workspace/MMEA-MAND/notebook/results/finished_runs_data_l49po2dj.csv
   파일 존재 여부: True

✅ 데이터 로드 완료: 15 runs
   • 컬럼 수: 251


In [34]:
# =============================================================================
# [셀 2. Sweep Name 파싱]
# =============================================================================
# sweep_name 형식: "Evaluate {model} ({modality}) Method on {dataset} (Seeds ...)"
# 예: "Evaluate TBN-MAND (All) Method on DataEgo (Seeds 1993-1997)"
#     → model_name_from_sweep: TBN-MAND
#     → modality_from_sweep: All
#     → dataset_from_sweep: DataEgo
#
# • 정규식으로 추출하여 df에 3개 컬럼 추가
# • 이후 create_ood_result_table에서 model_name_from_sweep, modality_from_sweep, dataset_from_sweep로 필터링
# =============================================================================

def parse_sweep_name(sweep_name):
    if pd.isna(sweep_name):
        return None, None, None
    pattern = r"Evaluate (.+?) \((.+?)\) Method on (.+?) \(Seeds"
    match = re.search(pattern, sweep_name)
    if match:
        return match.group(1).strip(), match.group(2).strip(), match.group(3).strip()
    return None, None, None

df[['model_name_from_sweep', 'modality_from_sweep', 'dataset_from_sweep']] = df['sweep_name'].apply(
    lambda x: pd.Series(parse_sweep_name(x))
)

print("✅ Sweep Name 파싱 완료")

✅ Sweep Name 파싱 완료


In [35]:
# =============================================================================
# [컬럼 목록 확인] df의 모든 컬럼명 출력
# =============================================================================
# • run_id, config, summary, Task/[xx-xx]_acc, Average_OOD/..., model_name_from_sweep 등
# • 테이블 생성 시 사용하는 컬럼 존재 여부 확인용
# =============================================================================
df.keys()


Index(['run_id', 'run_name', 'state', 'sweep_id', 'sweep_name', 'created_at',
       'lr', 'arch', 'host', 'mode',
       ...
       'Task/[06-07]_acc', 'Task/[08-09]_acc', 'Task/[10-11]_acc',
       'Task/[12-13]_acc', 'Task/[14-15]_acc', 'Task/[16-17]_acc',
       'Task/[18-19]_acc', 'model_name_from_sweep', 'modality_from_sweep',
       'dataset_from_sweep'],
      dtype='object', length=254)

In [36]:
# =============================================================================
# [셀 3. 테이블 생성 함수 정의]
# =============================================================================
#
# create_ood_result_table(df, ...):
#   • df를 datasets, increments, modality, seeds, fusion_type, cl_methods, ood_methods로 필터링
#   • 각 (CL_Method, OOD_Method) 조합별로:
#     - ACC: Task/avg_acc 평균
#     - Forgetting: Task/forgetting 평균
#     - AUC: Average_OOD/{ood_method}_auroc 평균
#     - FPR95: Average_OOD/{ood_method}_fpr95 평균
#   • fusion_type이 mand_fusion/auxiliary면 Hybrid 방법도 포함, 아니면 Baseline만
#   • MultiIndex 컬럼: (Dataset (increment: N), ACC/Forgetting/AUC/FPR95)
#
# save_ood_table_to_excel(result_df, filename):
#   • result_df를 엑셀로 저장, 헤더 스타일링(색상, 정렬, 열 너비) 적용
# =============================================================================

import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.cell.cell import MergedCell
from openpyxl.utils import get_column_letter

def create_ood_result_table(df, datasets=['DataEgo', 'UESTC-MMEA-CL'], increments=[2],
                            modality='All', seeds=[1993, 1994, 1995, 1996, 1997],
                            cl_methods=None, ood_methods=None, fusion_type=None):
    """OOD 결과를 표 형태로 생성"""
    if cl_methods is None:
        cl_methods = ['Replay'] # 'Replay', 'iCaRL', 'EWC', 'LwF'
    
    if ood_methods is None:
        # 기본 baseline methods (모든 fusion_type에서 사용 가능)
        baseline_methods = [
            'MSP_Baseline', 'MaxLogit_Baseline', 'ODIN_Baseline', 'Entropy_Baseline', 
            'Energy_Baseline', 'ReAct_Baseline', 'ASH_S_Baseline', 'Scale_Baseline', 'LTS_Baseline'
        ]
        
        # Hybrid methods (mand_fusion, auxiliary_head 등 aux logits 사용 시 - 1.dataego-mand-herding.yaml 대응)
        hybrid_methods = [
            # Hybrid UniformSum (Main+Aux, 각 모달리티 1:1:1)
            'MSP_Hybrid_UniformSum', 'MaxLogit_Hybrid_UniformSum', 'ODIN_Hybrid_UniformSum',
            'Entropy_Hybrid_UniformSum', 'Energy_Hybrid_UniformSum',
            # Hybrid UniformAverage (Main 1 + Aux 각 1/N)
            'MSP_Hybrid_UniformAverage', 'MaxLogit_Hybrid_UniformAverage', 'ODIN_Hybrid_UniformAverage',
            'Entropy_Hybrid_UniformAverage', 'Energy_Hybrid_UniformAverage',
            # Hybrid ConfRaw (raw confidence 가중)
            'MSP_Hybrid_ConfRaw', 'MaxLogit_Hybrid_ConfRaw', 'ODIN_Hybrid_ConfRaw',
            'Entropy_Hybrid_ConfRaw', 'Energy_Hybrid_ConfRaw',
            # Hybrid ConfNormalized (normalized confidence 가중)
            'MSP_Hybrid_ConfNormalized', 'MaxLogit_Hybrid_ConfNormalized', 'ODIN_Hybrid_ConfNormalized',
            'Entropy_Hybrid_ConfNormalized', 'Energy_Hybrid_ConfNormalized',
        ]
        
        # fusion_type에 따라 OOD 방법론 결정 (auxiliary, mand_fusion = Hybrid 지원)
        if fusion_type is not None:
            ft_lower = fusion_type.lower()
            if 'auxiliary' in ft_lower or 'mand_fusion' in ft_lower:
                ood_methods = baseline_methods + hybrid_methods
                print(f"📊 '{fusion_type}' → Baseline + Hybrid ({len(ood_methods)} methods)")
            else:
                ood_methods = baseline_methods
                print(f"📊 '{fusion_type}' → Baseline only ({len(ood_methods)} methods)")
        else:
            # fusion_type이 명시되지 않은 경우: 모든 방법론 사용 (기본값)
            ood_methods = baseline_methods + hybrid_methods
            print(f"⚠️  Fusion type not specified → Using all methods ({len(ood_methods)} total)")
            print(f"   💡 Tip: Specify 'fusion_type' parameter to filter methods appropriately")
    
    results = []
    print("\n" + "="*80)
    print("🔍 데이터 필터링 상세 정보")
    print("="*80)
    
    for cl_method in cl_methods:
        for ood_method in ood_methods:
            row_data = {'CL_Method': cl_method, 'OOD_Method': ood_method}
            
            for dataset in datasets:
                for increment in increments:
                    filtered = df[
                        (df['model_name_from_sweep'].str.contains(cl_method, na=False)) &
                        (df['modality_from_sweep'] == modality) &
                        (df['dataset_from_sweep'] == dataset)
                    ]
                    if 'increment' in df.columns:
                        filtered = filtered[filtered['increment'] == increment]
                    if 'seed' in df.columns:
                        filtered = filtered[filtered['seed'].isin(seeds)]
                    if fusion_type is not None and 'fusion_type' in df.columns:
                        filtered = filtered[filtered['fusion_type'] == fusion_type]
                    
                    # 📊 필터링 결과 출력
                    print(f"📌 CL={cl_method:12s} | OOD={ood_method:30s} | Dataset={dataset:20s} | Inc={increment:2d} → {len(filtered):3d} records")
                    
                    prefix = f"{dataset}_{increment}"
                    
                    # CL 메트릭
                    acc_col = 'Task/avg_acc'
                    row_data[f'{prefix}_ACC'] = filtered[acc_col].mean() if acc_col in filtered.columns and len(filtered) > 0 else np.nan
                    # Forgetting (CL 평가지표, Trainer에서 Task/forgetting 로깅 필요)
                    fgt_col = 'Task/forgetting'
                    row_data[f'{prefix}_Forgetting'] = filtered[fgt_col].mean() if fgt_col in filtered.columns and len(filtered) > 0 else np.nan
                    
                    # OOD 메트릭: ACC, AUC, FPR95만 (AUPR 제외)
                    for metric_name, metric_key in [('AUC', 'auroc'), ('FPR95', 'fpr95')]:
                        col = f'Average_OOD/{ood_method}_{metric_key}'
                        row_data[f'{prefix}_{metric_name}'] = filtered[col].mean() if col in filtered.columns and len(filtered) > 0 else np.nan
            
            results.append(row_data)
    
    print("="*80 + "\n")
    
    result_df = pd.DataFrame(results)
    
    # 3단계 MultiIndex 컬럼 생성 (Data 이름, increment는 config 동적)
    # Level 0: {Dataset} (increment {N})
    # Level 1: Known Classification | Unknown Detection
    # Level 2: ACC | Forgetting | AUC | FPR95
    KNOWN_METRICS = ('ACC', 'Forgetting')
    UNKNOWN_METRICS = ('AUC', 'FPR95')
    
    new_columns = []
    for col in result_df.columns:
        if col in ['CL_Method', 'OOD_Method']:
            new_columns.append(('', '', col))
        else:
            parts = col.rsplit('_', 1)
            if len(parts) == 2:
                dataset_inc, metric = parts[0], parts[1]
                ds_parts = dataset_inc.rsplit('_', 1)
                if len(ds_parts) == 2:
                    ds_name, inc_val = ds_parts[0], ds_parts[1]
                    level0 = f'{ds_name} (increment {inc_val})'
                    if metric in KNOWN_METRICS:
                        level1 = 'Known Classification'
                    elif metric in UNKNOWN_METRICS:
                        level1 = 'Unknown Detection'
                    else:
                        level1 = ''
                    new_columns.append((level0, level1, metric))
                else:
                    new_columns.append((dataset_inc, '', metric))
            else:
                new_columns.append(('', '', col))
    
    result_df.columns = pd.MultiIndex.from_tuples(new_columns)
    return result_df


def save_ood_table_to_excel(result_df, filename, apply_formatting=True):
    """OOD 결과 테이블을 엑셀로 저장 (3단계 헤더 + 연한 초록 스타일)"""
    result_df.to_excel(filename, merge_cells=True)
    
    if apply_formatting:
        wb = openpyxl.load_workbook(filename)
        ws = wb.active
        
        # 3단계 헤더용 스타일 (이미지: 연한 초록 배경, 굵은 검정 테두리)
        header_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  # 연한 초록
        header_font = Font(bold=True, size=11)
        thick_border = Border(
            left=Side(style='medium'), right=Side(style='medium'),
            top=Side(style='medium'), bottom=Side(style='medium')
        )
        thin_border = Border(
            left=Side(style='thin'), right=Side(style='thin'),
            top=Side(style='thin'), bottom=Side(style='thin')
        )
        center = Alignment(horizontal="center", vertical="center")
        
        # 3단계 헤더 행 스타일링 (Level 0, 1, 2 = Row 1, 2, 3)
        n_header_rows = result_df.columns.nlevels
        for row_idx in range(1, n_header_rows + 1):
            for cell in ws[row_idx]:
                if isinstance(cell, MergedCell):
                    continue
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = center
                cell.border = thick_border
        
        # CL Method, OOD Method, 데이터 셀
        data_start_row = n_header_rows + 1
        method_fill = PatternFill(start_color="E2EFDA", end_color="E2EFDA", fill_type="solid")
        for row in ws.iter_rows(min_row=data_start_row):
            for cell in row:
                if isinstance(cell, MergedCell):
                    continue
                cell.alignment = center
                cell.border = thin_border
                if 2 <= cell.column <= 3:
                    cell.fill = method_fill
                if cell.column > 2 and isinstance(cell.value, (int, float)):
                    cell.number_format = '0.0000'
        
        # 열 너비
        ws.column_dimensions['A'].width = 5
        ws.column_dimensions['B'].width = 12
        ws.column_dimensions['C'].width = 18
        for col_idx in range(4, ws.max_column + 1):
            ws.column_dimensions[get_column_letter(col_idx)].width = 12
        
        wb.save(filename)
    
    print(f"✅ 엑셀 저장 완료: {filename}")
    print(f"   • {result_df.shape[0]} rows × {result_df.shape[1]} columns")

print("✅ 함수 로드 완료")


✅ 함수 로드 완료


In [37]:
# =============================================================================
# [셀 4. 테이블 생성 및 저장]
# =============================================================================
# • cfg에서 target_datasets, target_increments, target_modality, target_seeds,
#   target_fusion_types, cl_methods, ood_methods 추출
# • fusion_type × increment 조합마다:
#   1) create_ood_result_table() 호출 → ood_table
#   2) display(ood_table.head(5)) 미리보기
#   3) save_ood_table_to_excel()로 엑셀 저장
# • 출력 파일명: v1_ood_results_{Dataset}_inc{N}_{modality}_{fusion_type}.xlsx
# =============================================================================

target_datasets = cfg['target_datasets']
target_increments = cfg['target_increments']
target_modality = cfg['target_modality']
target_seeds = cfg['target_seeds']
target_fusion_types = cfg['target_fusion_types']
cl_methods = cfg['cl_methods']
ood_methods = cfg.get('ood_methods')  # None이면 create_ood_result_table에서 fusion_type으로 자동

output_dir = notebook_dir / 'results'
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("🔄 OOD 결과 테이블 생성")
print("=" * 100)
print(f"  • 데이터셋: {target_datasets} | Increment: {target_increments}")
print(f"  • Fusion: {target_fusion_types} | CL: {cl_methods}")
print()

for fusion_type in target_fusion_types:
    for increment in target_increments:
        print("\n" + "=" * 100)
        print(f"📊 {fusion_type}, inc={increment}")
        print("=" * 100)
        
        version_prefix = 'v2' if fusion_type == 'auxiliary_head_v2_7' else 'v1'
        dataset_str = '-'.join(target_datasets)
        output_filename = output_dir / f'{version_prefix}_ood_results_{dataset_str}_inc{increment}_{target_modality}_{fusion_type}.xlsx'
        
        ood_table = create_ood_result_table(
            df=df,
            datasets=target_datasets,
            increments=[increment],
            modality=target_modality,
            seeds=target_seeds,
            fusion_type=fusion_type,
            cl_methods=cl_methods,
            ood_methods=ood_methods
        )
        
        print(f"\n✅ 테이블 생성 완료: {ood_table.shape[0]} rows × {ood_table.shape[1]} columns")
        
        # 미리보기
        print("\n📋 테이블 미리보기 (처음 5행)")
        print("-" * 100)
        display(ood_table.head(5))
        
        # 엑셀 저장
        print("\n💾 엑셀 파일로 저장 중...")
        save_ood_table_to_excel(ood_table, str(output_filename), apply_formatting=True)
        
        print(f"✅ 완료! 파일: {output_filename}")

print("\n" + "=" * 100)
print(f"🎉 모든 작업 완료! 총 {len(target_fusion_types) * len(target_increments)}개의 엑셀 파일이 생성되었습니다.")
print("=" * 100)


🔄 OOD 결과 테이블 생성
  • 데이터셋: ['DataEgo'] | Increment: [10, 5, 2]
  • Fusion: ['mand_fusion'] | CL: ['MAND']


📊 mand_fusion, inc=10

🔍 데이터 필터링 상세 정보
📌 CL=MAND         | OOD=MaxLogit_Baseline              | Dataset=DataEgo              | Inc=10 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Dataset=DataEgo              | Inc=10 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Dataset=DataEgo              | Inc=10 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_ConfNormalized | Dataset=DataEgo              | Inc=10 →   5 records


✅ 테이블 생성 완료: 4 rows × 6 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment 10)             \
                                              Known Classification              
  CL_Method                      OOD_Method                    ACC Forgetting   
0      MAND               MaxLogit_Baseline                 90.898      4.862   
1      MAND      MaxLogit_Hybrid_UniformSum                 90.898      4.862   
2      MAND  MaxLogit_Hybrid_UniformAverage                 90.898      4.862   
3      MAND  MaxLogit_Hybrid_ConfNormalized                 90.898      4.862   

                                
  Unknown Detection             
                AUC      FPR95  
0         80.892484  72.713178  
1         78.792046  70.232558  
2         79.892147  71.782946  
3         78.035726  76.744186


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc10_All_mand_fusion.xlsx
   • 4 rows × 6 columns
✅ 완료! 파일: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc10_All_mand_fusion.xlsx

📊 mand_fusion, inc=5

🔍 데이터 필터링 상세 정보
📌 CL=MAND         | OOD=MaxLogit_Baseline              | Dataset=DataEgo              | Inc= 5 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Dataset=DataEgo              | Inc= 5 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Dataset=DataEgo              | Inc= 5 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_ConfNormalized | Dataset=DataEgo              | Inc= 5 →   5 records


✅ 테이블 생성 완료: 4 rows × 6 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment 5)             \
                                             Known Classification              
  CL_Method                      OOD_Method                   ACC Forgetting   
0      MAND               MaxLogit_Baseline                89.014   5.770667   
1      MAND      MaxLogit_Hybrid_UniformSum                89.014   5.770667   
2      MAND  MaxLogit_Hybrid_UniformAverage                89.014   5.770667   
3      MAND  MaxLogit_Hybrid_ConfNormalized                89.014   5.770667   

                                
  Unknown Detection             
                AUC      FPR95  
0         78.915551  77.105797  
1         76.136003  78.968060  
2         77.819269  77.790912  
3         73.757214  82.003334


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc5_All_mand_fusion.xlsx
   • 4 rows × 6 columns
✅ 완료! 파일: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc5_All_mand_fusion.xlsx

📊 mand_fusion, inc=2

🔍 데이터 필터링 상세 정보
📌 CL=MAND         | OOD=MaxLogit_Baseline              | Dataset=DataEgo              | Inc= 2 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformSum     | Dataset=DataEgo              | Inc= 2 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_UniformAverage | Dataset=DataEgo              | Inc= 2 →   5 records
📌 CL=MAND         | OOD=MaxLogit_Hybrid_ConfNormalized | Dataset=DataEgo              | Inc= 2 →   5 records


✅ 테이블 생성 완료: 4 rows × 6 columns

📋 테이블 미리보기 (처음 5행)
----------------------------------------------------------------------------------------------------


DataEgo (increment 2)             \
                                             Known Classification              
  CL_Method                      OOD_Method                   ACC Forgetting   
0      MAND               MaxLogit_Baseline                90.078   6.500222   
1      MAND      MaxLogit_Hybrid_UniformSum                90.078   6.500222   
2      MAND  MaxLogit_Hybrid_UniformAverage                90.078   6.500222   
3      MAND  MaxLogit_Hybrid_ConfNormalized                90.078   6.500222   

                                
  Unknown Detection             
                AUC      FPR95  
0         73.253546  81.221381  
1         69.250843  87.436709  
2         71.957665  85.365564  
3         69.584205  86.051677


💾 엑셀 파일로 저장 중...
✅ 엑셀 저장 완료: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc2_All_mand_fusion.xlsx
   • 4 rows × 6 columns
✅ 완료! 파일: /workspace/MMEA-MAND/notebook/results/v1_ood_results_DataEgo_inc2_All_mand_fusion.xlsx

🎉 모든 작업 완료! 총 3개의 엑셀 파일이 생성되었습니다.


In [38]:
# =============================================================================
# [데이터 진단] cfg 설정에 맞는 데이터가 실제로 있는지 확인
# =============================================================================
# • 1) increment 분포: df에 몇 개의 increment 값이 있는지
# • 2) fusion_type 분포: mand_fusion, concat 등
# • 3) cfg 조건별 record 수: (CL, Dataset, inc, fusion_type) 조합마다 몇 건인지
#   → 0건이면 테이블에 NaN으로 나오므로, sweep/실험 설정 확인 필요
# =============================================================================

print("=" * 100)
print(f"🔍 데이터 진단 (preset={preset}, fusion={cfg['target_fusion_types']}, inc={cfg['target_increments']})")
print("=" * 100)

print("\n1️⃣ increment 분포:")
print(df['increment'].value_counts().sort_index() if 'increment' in df.columns else "(컬럼 없음)")

print("\n2️⃣ fusion_type 분포:")
print(df['fusion_type'].value_counts() if 'fusion_type' in df.columns else "(컬럼 없음)")

print("\n3️⃣ 현재 cfg 조건에 맞는 데이터:")
req_cols = ['increment', 'fusion_type', 'model_name_from_sweep', 'modality_from_sweep', 'dataset_from_sweep']
if all(c in df.columns for c in req_cols):
    for ft in cfg['target_fusion_types']:
        for inc in cfg['target_increments']:
            for cl in cfg['cl_methods']:
                for ds in cfg['target_datasets']:
                    f = df[(df['increment']==inc) & (df['fusion_type']==ft) &
                           (df['model_name_from_sweep'].str.contains(cl, na=False)) &
                           (df['modality_from_sweep']==cfg['target_modality']) &
                           (df['dataset_from_sweep']==ds)]
                    print(f"   {cl:8s} | {ds:18s} | inc={inc:2d} | {ft:18s} → {len(f):3d} records")
else:
    print("   (필요 컬럼 부족)")

print("\n" + "=" * 100)

🔍 데이터 진단 (preset=dataego_mand, fusion=['mand_fusion'], inc=[10, 5, 2])

1️⃣ increment 분포:
2     5
5     5
10    5
Name: increment, dtype: int64

2️⃣ fusion_type 분포:
mand_fusion    15
Name: fusion_type, dtype: int64

3️⃣ 현재 cfg 조건에 맞는 데이터:
   MAND     | DataEgo            | inc=10 | mand_fusion        →   5 records
   MAND     | DataEgo            | inc= 5 | mand_fusion        →   5 records
   MAND     | DataEgo            | inc= 2 | mand_fusion        →   5 records



In [39]:
# =============================================================================
# [추가 확인] sweep_name 파싱 결과 - model_name_from_sweep 고유값
# =============================================================================
# • parse_sweep_name으로 추출된 모델명 목록 (예: TBN-MAND, TBN-Replay 등)
# • cl_methods 필터와 일치하는지 확인용
# =============================================================================
df['model_name_from_sweep'].unique()

array(['TBN-MAND'], dtype=object)